In [ ]:
#import relevant libraries
import os

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import dabest

import NLCLIMB_asyn
import NLMATH_asyn

#NOTE: SUPPRESSES WARNINGS!

import warnings

warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", category=UserWarning)

In [ ]:
metricframe_cache = {}

def metricframe(genotype):
    if genotype in metricframe_cache:
        return metricframe_cache[genotype].copy()

    df = pd.read_csv(openPath + genotype + ".csv")
    df = df.drop(df.columns[[0]], axis = 1)   #the compiled CSVs carry an index column
    df = NLCLIMB_asyn.generation(df, genotype)

    cy = NLMATH_asyn.calcgraph(df, "Y.*")
    cv = NLMATH_asyn.calcgraph(df, "Velocity.*")
    cb = NLMATH_asyn.calcgraph(NLMATH_asyn.boutspeed(df), "BSpeed.*")
    si = NLMATH_asyn.straightnessindexmeter(df, genotype)

    out = pd.DataFrame()
    for n in phase:
        h = cy[cy.ExperimentState == n].iloc[:, 2:].mean(axis=0)
        v = cv[cv.ExperimentState == n].iloc[:, 2:].mean(axis=0)
        b = cb[cb.ExperimentState == n].iloc[:, 2:].mean(axis=0)
        s = si[si.ExperimentState == n]["averagestraightnessindex"]

        t = pd.DataFrame({"Height": h.values, "Speed": v.values,
                          "BSpeed": b.values, "SI": s.values})
        t["fly"] = [genotype + "_" + c.rsplit("_", 1)[-1] for c in h.index]
        t["Phase"] = n
        out = pd.concat([out, t])

    out = out.reset_index(drop=True)
    metricframe_cache[genotype] = out

    return out.copy()


def pooled(genotype, metric):
    #one value per fly, the mean of its three phases. A fly the tracker lost in one phase
    #leaves altogether, so every fly is averaged over the same 90 s.
    f = metricframe(genotype).dropna(subset = [metric])
    f = f.groupby("fly").filter(lambda g: len(g) == len(phase))
    f = f.groupby("fly", sort = False)[metric].mean().reset_index()

    #the colour key: a control keeps one colour at every age, a cross is shaded by its age
    name, age = genotype.rsplit("_", 1)
    name = name.replace(" x ", " > ")
    return f.assign(Source = name if name.startswith("w1118") else name + " " + age)


def parents(cross, age):
    #the parental controls of one cross, pooled as plot 2 of 6. Forestplot pools them.
    #w1118 x elav for most crosses; elav x 95240 also has w1118 x 95240.
    driver, responder = cross.split(" x ", 1)
    return [g for g in ["w1118 x " + driver, "w1118 x " + responder] if g + "_" + age in listofgenotypes]


#PLOT A - at each age, each cross against its own parental controls. Unpaired, one pair per cross.
#dabest will not reuse a group across pairs, so each control is named after its cross.
#Returns the dabest object per age, for the raw + contrast figure, and the contrasts as rows.

def crosscontrasts(metric):
    dbs, rows = {}, []

    for age in ages:
        data = []
        for cross in crosses:
            if cross + "_" + age not in listofgenotypes or not parents(cross, age):
                continue
            wt = pd.concat([pooled(g + "_" + age, metric) for g in parents(cross, age)])
            expt = pooled(cross + "_" + age, metric)
            if wt.fly.nunique() < minflies or expt.fly.nunique() < minflies:
                print(cross, age, "is", wt.fly.nunique(), "against", expt.fly.nunique(), "flies, skipped")
                continue
            data += [wt.assign(Group = "Control " + cross.split(" x ", 1)[1]),
                     expt.assign(Group = cross.replace(" x ", " > "))]

        if not data:
            continue

        data = pd.concat(data).reset_index(drop=True)
        groups = list(pd.unique(data.Group))
        db = dabest.load(data = data, idx = tuple(zip(groups[::2], groups[1::2])), x = "Group", y = metric,
                         resamples = resamples)
        dbs[age] = db

        for _, r in db.hedges_g.results.iterrows():
            rows.append({"block": age, "group": r["control"], "difference": np.nan, "low": np.nan,
                         "high": np.nan, "bootstraps": None, "N": r["control_N"]})
            rows.append({"block": age, "group": r["test"], "difference": r["difference"],
                         "low": r["bca_low"], "high": r["bca_high"],
                         "bootstraps": r["bootstraps"], "N": r["test_N"]})

    return dbs, pd.DataFrame(rows)


#PLOT B - one cross per row, control against experimental at each age, as plot 2 of 6. Forestplot.

def agecontrasts(metric):
    dbs, rows = {}, []

    for cross in crosses:
        data = []
        for age in ages:
            if cross + "_" + age not in listofgenotypes or not parents(cross, age):
                continue
            wt = pd.concat([pooled(g + "_" + age, metric) for g in parents(cross, age)])
            expt = pooled(cross + "_" + age, metric)
            if wt.fly.nunique() < minflies or expt.fly.nunique() < minflies:
                print(cross, age, "is", wt.fly.nunique(), "against", expt.fly.nunique(), "flies, skipped")
                continue
            data += [wt.assign(Group = "Control " + age), expt.assign(Group = "Expt " + age)]

        if not data:
            continue

        data = pd.concat(data).reset_index(drop=True)
        idx = tuple((g, "Expt " + g.split(" ")[1]) for g in pd.unique(data.Group) if g.startswith("Control"))
        db = dabest.load(data = data, idx = idx, x = "Group", y = metric, resamples = resamples)
        dbs[cross] = db

        for _, r in db.hedges_g.results.iterrows():
            rows.append({"block": cross, "group": r["control"], "difference": np.nan, "low": np.nan,
                         "high": np.nan, "bootstraps": None, "N": r["control_N"]})
            rows.append({"block": cross, "group": r["test"], "difference": r["difference"],
                         "low": r["bca_low"], "high": r["bca_high"],
                         "bootstraps": r["bootstraps"], "N": r["test_N"]})

    return dbs, pd.DataFrame(rows)


def dabestpanel(db, ax, m):
    #dabest crashes given both color_col and a custom_palette dict, so the colours go in
    #through seaborn, ordered the way dabest walks the colour column - as in 4. Dabest
    order = list(pd.unique(db._plot_data["Source"]))
    sns.set_palette([colourof(o) for o in order])

    db.hedges_g.plot(ax = ax, color_col = "Source",
                     raw_label = rawlabels[m], contrast_label = "Hedges' g",
                     raw_desat = 1, contrast_desat = 1,
                     raw_ylim = rawylims[m], contrast_ylim = contrastylims[m],
                     fontsize_rawxlabel = 8, fontsize_contrastxlabel = 8)


def contrastmark(ax, at, r, colour):
    #the right half of the bootstrap distribution, clipped the way the DrosoClimb forest plot clips it
    parts = ax.violinplot(r["bootstraps"], positions = [at], widths = 0.5,
                          showextrema = False, showmedians = False)
    for pc in parts["bodies"]:
        v = pc.get_paths()[0].vertices
        v[:, 0] = np.clip(v[:, 0], np.mean(v[:, 0]), np.inf)
        pc.set_facecolor(colour)
        pc.set_alpha(0.4)

    ax.plot([at, at], [r["low"], r["high"]], color = "black", lw = 1, zorder = 1)
    ax.plot(at, r["difference"], "o", color = colour, markersize = 5, zorder = 2,
            markeredgecolor = "black", markeredgewidth = 0.5)
    ax.text(at - 0.18, r["difference"], "%+.2f" % r["difference"],
            ha = "right", va = "center", fontsize = 7)


def tidy(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(labelsize = 8)

In [ ]:
#initial file processing

laptop = "C:\\Users\\Nicole Lee\\"
homecomp = "C:\\Users\\user\\"
labcomp = "C:\\Users\\User\\"

path2 = "NUS Dropbox\\acclab\\Nicole M Lee\\"
openPath = homecomp + path2 + "PD\\Data Compilation\\"
savefiglocation = openPath + "images\\"

isExist = os.path.exists(savefiglocation)
if not isExist:
    os.makedirs(savefiglocation)

phase = ["First phase", "Second phase", "Third phase"]
ages = ["D10", "D20", "D30"]
metrics = ["Height", "Speed"]
rawlabels = {"Height": "Height climbed (mm)", "Speed": "Overall speed (mm/s)"}

#the crosses to plot, and the smallest arm to take a contrast from. Edit these.
crosses = ["elav x 8146", "elav x 8147", "elav x 51375", "elav x 51376", "elav x 95240"]
minflies = 5

#the dabest default, so the intervals match the ones 4. Dabest draws
resamples = 5000

#each responder has a hue, shaded light to dark with age. Edit these.
shades = {"blue":  {"D10": "#9ECAE1", "D20": "#4292C6", "D30": "#08306B"},
          "green": {"D10": "#90C590", "D20": "#228B22", "D30": "#114611"},
          "pink":  {"D10": "#FBB4C9", "D20": "#E7298A", "D30": "#91003F"}}
responderhue = {"8146": "blue", "95240": "blue", "51375": "green", "51376": "green", "8147": "pink"}

#the pooled controls keep their own marker colours, as in 4. Dabest
controlcolours = {"w1118 > elav": "#000000", "w1118 > 95240": "#8B4513"}
grey = "#808080"

def colourof(source):
    #a source is a control ("w1118 > elav") or a cross with its age ("elav > 8146 D10")
    if source.startswith("w1118"):
        return controlcolours.get(source, grey)
    cross, age = source.rsplit(" ", 1)
    return shades[responderhue[cross.split(" > ")[-1]]][age]

listofgenotypes = sorted(f[:-4] for f in os.listdir(openPath) if f.endswith(".csv"))

print(len(listofgenotypes), "genotypes")

In [ ]:
#the contrasts, computed once per metric for all four figures below

resA = {m: crosscontrasts(m) for m in metrics}
resB = {m: agecontrasts(m) for m in metrics}

In [ ]:
#PLOT A - dabest raw and contrast, one row per age, each cross against its own control

mpl.rcParams['svg.fonttype'] = 'none'

#axis ranges, per metric. Edit these.
rawylims = {"Height": (0, 90), "Speed": (0, 15)}
contrastylims = {"Height": (-2, 2), "Speed": (-2, 2)}

for m in metrics:
    dbs, res = resA[m]

    fig, axes = plt.subplots(len(ages), 1, figsize = (18, 18), gridspec_kw = {"hspace": 0.5})

    for ax, age in zip(axes, ages):
        if age not in dbs:
            ax.set_axis_off()
            continue
        dabestpanel(dbs[age], ax, m)
        ax.set_title(age, loc = "left", fontweight = "bold", fontsize = 13)

    fig.suptitle("Pooled phases, Hedges' g against the control: " + rawlabels[m],
                 fontweight = 'bold', fontsize = 14, y = 0.93)
    #fig.savefig(savefiglocation + "Forest_pooled_crosses_raw_" + m + ".svg", bbox_inches='tight')

    print(savefiglocation + "Forest_pooled_crosses_raw_" + m + ".svg")

In [ ]:
#PLOT A2 - the contrasts of plot A alone

mpl.rcParams['svg.fonttype'] = 'none'

#axis range, per metric. Edit these.
contrastylims = {"Height": (-2, 2), "Speed": (-2, 2)}

for m in metrics:
    dbs, res = resA[m]

    fig, axes = plt.subplots(len(ages), 1, figsize = (12, 10), gridspec_kw = {"hspace": 0.7})

    for ax, age in zip(axes, ages):
        tidy(ax)
        ax.set_ylim(contrastylims[m])
        ax.axhline(0, color = "black", lw = 0.5)
        ax.set_ylabel("Hedges' g", fontsize = 10)
        ax.text(-0.1, 0.5, age, transform = ax.transAxes, rotation = 90,
                ha = "center", va = "center", fontweight = "bold", fontsize = 13)

        if age not in dbs:
            ax.set_xticks([])
            continue

        here = res[res.block == age]
        groups = list(pd.unique(dbs[age]._plot_data["Group"]))

        #the line from each control's baseline to its experimental
        for _, r in here.dropna(subset = ["difference"]).iterrows():
            at = groups.index(r["group"])
            colour = colourof(r["group"] + " " + age)
            ax.plot([at - 1, at], [0, r["difference"]], color = colour, lw = 0.8, alpha = 0.6, zorder = 0)
            contrastmark(ax, at, r, colour)

        ns = here.set_index("group")["N"]
        ax.set_xlim(-0.6, len(groups) - 0.4)
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([g.replace("Control ", "Control\n") + "\nN = " + str(ns[g]) for g in groups], fontsize = 7)

    fig.suptitle("Pooled phases, Hedges' g against the control: " + rawlabels[m],
                 fontweight = 'bold', fontsize = 14, y = 0.97)
    #fig.savefig(savefiglocation + "Forest_pooled_crosses_" + m + ".svg", bbox_inches='tight')

    print(savefiglocation + "Forest_pooled_crosses_" + m + ".svg")

In [ ]:
#PLOT B - dabest raw and contrast, one row per cross, control against experimental at each age

mpl.rcParams['svg.fonttype'] = 'none'

#axis ranges, per metric. Edit these.
rawylims = {"Height": (0, 90), "Speed": (0, 15)}
contrastylims = {"Height": (-2, 2), "Speed": (-2, 2)}

for m in metrics:
    dbs, res = resB[m]

    fig, axes = plt.subplots(len(crosses), 1, figsize = (10, 6 * len(crosses)), gridspec_kw = {"hspace": 0.5})

    for ax, cross in zip(axes, crosses):
        if cross not in dbs:
            ax.set_axis_off()
            continue
        dabestpanel(dbs[cross], ax, m)
        ax.set_title(cross.replace(" x ", " > "), loc = "left", fontweight = "bold", fontsize = 13)

    fig.suptitle("Pooled phases, Hedges' g against the control: " + rawlabels[m],
                 fontweight = 'bold', fontsize = 14, y = 0.9)
    #fig.savefig(savefiglocation + "Forest_pooled_ages_raw_" + m + ".svg", bbox_inches='tight')

    print(savefiglocation + "Forest_pooled_ages_raw_" + m + ".svg")

In [ ]:
#PLOT B2 - the contrasts of plot B alone

mpl.rcParams['svg.fonttype'] = 'none'

#axis range, per metric. Edit these.
contrastylims = {"Height": (-2, 2), "Speed": (-2, 2)}

groups = [g + " " + a for a in ages for g in ["Control", "Expt"]]

for m in metrics:
    dbs, res = resB[m]

    fig, axes = plt.subplots(len(crosses), 1, figsize = (7, 2.6 * len(crosses)), gridspec_kw = {"hspace": 0.7})

    for ax, cross in zip(axes, crosses):
        tidy(ax)
        ax.set_ylim(contrastylims[m])
        ax.set_xlim(-0.6, len(groups) - 0.4)
        ax.axhline(0, color = "black", lw = 0.5)
        ax.set_ylabel(cross.replace(" x ", " > "), fontweight = 'bold', fontsize = 10)

        here = res[res.block == cross] if not res.empty else res
        if here.empty:
            ax.set_xticks([])
            continue

        #the line from each control's baseline to its experimental
        for _, r in here.dropna(subset = ["difference"]).iterrows():
            at = groups.index(r["group"])
            colour = colourof(cross.replace(" x ", " > ") + " " + r["group"].split(" ")[1])
            ax.plot([at - 1, at], [0, r["difference"]], color = colour, lw = 0.8, alpha = 0.6, zorder = 0)
            contrastmark(ax, at, r, colour)

        ns = here.set_index("group")["N"]
        ax.set_xticks(range(len(groups)))
        ax.set_xticklabels([g.replace(" ", "\n") + ("\nN = " + str(ns[g]) if g in ns.index else "")
                            for g in groups], fontsize = 7)

    fig.suptitle("Pooled phases, Hedges' g against the control: " + rawlabels[m],
                 fontweight = 'bold', fontsize = 13, y = 0.95)
    #fig.savefig(savefiglocation + "Forest_pooled_ages_" + m + ".svg", bbox_inches='tight')

    print(savefiglocation + "Forest_pooled_ages_" + m + ".svg")